# ASL MVP — training on Colab (accuracy-push revision)

Two paths through this notebook:
- **Fast path** (~40 min): improved 30-word training only → run cells 1-5a, then 6, 7, 8.
- **Max-accuracy path** (~3 h): **pretrain on all 250 words, then fine-tune on the 30** → run cells 1-4, 5b, 6-pre, 6ft, 7, 8. This is the biggest accuracy lever we own.

Before you start: **Runtime → Change runtime type → Hardware accelerator = T4 GPU → Save.**

## Cell 1 — confirm the GPU
Expect `Tesla T4`.

In [ ]:
!nvidia-smi

## Cell 2 — environment (Keras 2 shim + deps)
Expect a TF version print. The `tf_keras` shim routes `tf.keras` to the Keras 2 API our code needs (the standalone `keras.__version__` may still say 3.x — that's fine and expected).

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # route tf.keras -> Keras 2
!pip install -q tf-keras pandas pyarrow
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 3 — get the code
Upload `train.py` + `vocab_30.json` via the Files panel (left sidebar → upload icon), then run this to confirm. (The repo is private, so `git clone` won't work without auth — manual upload is simplest.)

In [ ]:
!ls -la train.py vocab_30.json

## Cell 4 — AWS credentials (read-only key)
Paste the `colab-readonly` access key + secret. Expect your account id + user arn at the end.

In [ ]:
import getpass, os
os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('AWS Access Key ID: ')
os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('AWS Secret Access Key: ')
os.environ['AWS_DEFAULT_REGION'] = 'eu-north-1'
!pip install -q awscli
!aws sts get-caller-identity

## Cell 5a — download data: 30 words only (~230 MB) — FAST PATH
Skip this if you're doing the max-accuracy path (5b downloads everything).

In [ ]:
import json, subprocess, os
os.makedirs('data', exist_ok=True)
words = json.load(open('vocab_30.json'))['words']
subprocess.run(['aws','s3','cp','s3://asl-mvp-dataset/splits/split_manifest.parquet','data/'], check=True)
for w in words:
    subprocess.run(['aws','s3','sync',f's3://asl-mvp-dataset/preprocessed/by_word/{w}',f'data/by_word/{w}'], check=True)
print('done. words downloaded:', len(words))
!du -sh data

## Cell 5b — download data: ALL 250 words (~2 GB) — MAX-ACCURACY PATH
Needed for the 250-class pretrain. Takes a few minutes; watch the sync lines.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
!aws s3 cp s3://asl-mvp-dataset/splits/split_manifest.parquet data/
!aws s3 sync s3://asl-mvp-dataset/preprocessed/by_word data/by_word --only-show-errors
!ls data/by_word | wc -l   # expect 250
!du -sh data

## Cell 6-pre — PRETRAIN on all 250 words (~2-2.5 h) — MAX-ACCURACY PATH
Teaches the backbone what signing looks like on ~76k sequences before it specializes on your 30. Keep the tab open (Colab free disconnects idle sessions). Expect the same epoch lines, 250 classes, slower epochs (~2 min each). Skip if fast path.

In [ ]:
!python train.py --data-dir ./data --fold 0 --all-words --epochs 60 --save-weights backbone_250.weights.h5

## Cell 6 — FAST PATH: improved 30-word training (~30-40 min)
Stronger augmentation + label smoothing + higher dropout vs. the first 0.688 run. Use this **or** 6-pre+6ft, not both.

In [ ]:
!python train.py --data-dir ./data --fold 0

## Cell 6ft — MAX-ACCURACY PATH: fine-tune the 30-word model from the pretrained backbone (~30-40 min)
Lower LR because the backbone already knows a lot.

In [ ]:
!python train.py --data-dir ./data --fold 0 --init-from artifacts/backbone_250.weights.h5 --lr 2e-4

## Cell 7 — result
Overall + per-word accuracy. Compare to the 0.688 baseline.

In [ ]:
import json
r = json.load(open('artifacts/eval_mvp30_fold0.json'))
print('overall val acc:', round(r['overall_acc'], 4), '| init_from:', r['init_from'])
print('worst 5 words :', r['worst_words'])
for w, a in sorted(r['per_word_acc'].items(), key=lambda x: x[1]):
    print(f'  {w:12s} {a:.3f}')

## Cell 8 — download artifacts to your laptop (DO THIS BEFORE CLOSING COLAB)
Everything in `/content` is lost when the session recycles.

In [ ]:
from google.colab import files
!zip -qr fold0_artifacts.zip artifacts
files.download('fold0_artifacts.zip')